# 03 - Train Classical Baselines

This notebook reuses the production training pipeline to report cross-validated metrics, visualize score summaries, and inspect feature importance for the CIC-IDS2017 elephant/mice task.

**Outputs**
- Table 3: averaged accuracy/precision/recall/F1 (means ± std. dev.) across 5 stratified group folds.
- Figure 7: bar chart of F1 scores with standard-deviation error bars.
- Figure 8: confusion matrices for the best and worst models.
- Figure 9: random-forest feature importance plot.
- Markdown summary interpreting the model behavior.

In [ ]:

from dataclasses import asdict
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay

plt.style.use("seaborn-v0_8-colorblind")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import CONFIG
from src.models.classical_baselines import FEATURE_COLUMNS, run_baselines


In [ ]:

aggregated_results, predictions = run_baselines()
aggregated_results


In [ ]:

metrics_table = pd.DataFrame(
    [
        {
            "Model": res.model,
            "Accuracy": f"{res.accuracy_mean:.4f} ± {res.accuracy_std:.4f}",
            "Precision": f"{res.precision_mean:.4f} ± {res.precision_std:.4f}",
            "Recall": f"{res.recall_mean:.4f} ± {res.recall_std:.4f}",
            "F1": f"{res.f1_mean:.4f} ± {res.f1_std:.4f}",
        }
        for res in aggregated_results
    ]
)
metrics_table


In [ ]:

fig, ax = plt.subplots(figsize=(8, 4))
models = [res.model for res in aggregated_results]
f1_means = [res.f1_mean for res in aggregated_results]
f1_stds = [res.f1_std for res in aggregated_results]
ax.bar(models, f1_means, yerr=f1_stds, color="#4c72b0", alpha=0.85, capsize=4)
ax.set_ylim(0, max(f1_means) * 1.2)
ax.set_ylabel("F1 score")
ax.set_title("Figure 7 – Cross-validated F1 scores (mean ± σ)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:

from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
best_model = aggregated_results[0].model
worst_model = aggregated_results[-1].model

def plot_confusion(ax, model_name):
    y_true, y_pred = predictions[model_name]
    ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred,
        display_labels=["Mice", "Elephant"],
        cmap="Blues",
        colorbar=False,
        ax=ax,
        values_format='d',
    )
    ax.set_title(f"{model_name} confusion matrix")

plot_confusion(axes[0], best_model)
plot_confusion(axes[1], worst_model)
plt.suptitle("Figure 8 – Best vs. worst model confusion matrices")
plt.tight_layout()
plt.show()


In [ ]:

data = pd.read_csv(CONFIG.paper_small_csv)
X = data[list(FEATURE_COLUMNS)]
y = data["target_traffic"].astype(int)
rf = RandomForestClassifier(n_estimators=200, random_state=CONFIG.random_state, n_jobs=-1)
rf.fit(X, y)
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLUMNS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(importances.index, importances.values, color="#55a868")
ax.set_ylabel("Importance")
ax.set_title("Figure 9 – Random forest feature importance")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


The 5-fold stratified group CV shows that tree-based models dominate under the constrained five-feature view, while Gaussian NB struggles with the skewed, non-Gaussian bytes distribution. Random forests place most weight on flow size and timing features, validating the paper’s focus on lightweight telemetry. The confusion matrices confirm that the best models keep both false positives and false negatives near zero despite the 0.09% elephant prior.